# Notebook 04 . Analyse sémantique (LDA + Sentence-CamemBERT)

## De quoi parle ce notebook ?

On explore ici la **structure thématique** des discours VSS a l'Assemblée Nationale avec deux familles de méthodes :

**Partie A . Topic modeling LDA** : on découvre automatiquement les thématiques latentes dans le corpus.
L'enjeu est de voir si un "topic identitaire/migratoire" émerge naturellement et comment son poids
varie selon les blocs politiques et les années.

**Partie B . Embeddings Sentence-CamemBERT** : on transforme chaque discours en un vecteur numérique
pour mesurer la similarité sémantique entre blocs au fil du temps.

### Pourquoi le CamemBERT "brut" ne fonctionne pas pour la similarité

Dans la version précédente de ce notebook, on utilisait CamemBERT-base avec le vecteur CLS
pour représenter les documents. Les résultats étaient décevants : similarité proche de 1.0 pour
tous les blocs, aucune variation exploitable.

C'est un problème **bien documenté dans la littérature**. Jacob Devlin (créateur de BERT) a
lui-meme déclaré : "BERT does not generate meaningful sentence vectors" (Reimers et Gurevych, 2019).
Le vecteur CLS d'un modèle BERT/CamemBERT **non fine-tuné** produit des représentations
dans un espace qui n'est pas adapté a la similarité cosinus.

La solution, établie par Reimers et Gurevych (2019) dans le papier **Sentence-BERT**, consiste
a fine-tuner le modèle sur des taches de similarité avec un réseau siamois. Pour le francais,
le modèle **Sentence-CamemBERT-Large** (La Javaness, 2022) fait exactement cela. Il produit
des embeddings de 1024 dimensions spécifiquement optimisés pour la similarité cosinus.

### Choix méthodologique pour la LDA

On ne retient pas aveuglément le k qui maximise le score de cohérence u_mass.
Un modèle a 3 topics a beau avoir le meilleur score, il est beaucoup trop grossier
pour capturer un signal aussi spécifique que le cadrage identitaire des VSS.
On impose un **plancher de 12 topics** et on sélectionne le meilleur score au-dessus de ce seuil,
conformément a la pratique recommandée par Mimno et al. (2011) pour les corpus thématiquement denses.

## Partie A . Topic Modeling LDA

### A.1 Imports et chargement

In [1]:
import pandas as pd
import numpy as np
import seaborn as sns
import re, os, pickle, warnings
from tqdm import tqdm

from gensim import corpora, models
from gensim.models import LdaModel, LdaMulticore, Phrases
from gensim.models.coherencemodel import CoherenceModel

import nltk
from nltk.corpus import stopwords
from nltk.stem.snowball import FrenchStemmer
nltk.download('stopwords', quiet=True)

warnings.filterwarnings('ignore')
stemmer = FrenchStemmer()

import matplotlib.pyplot as plt
import matplotlib as mpl

PALETTE_PASTEL = ["#A8D8EA", "#AA96DA", "#FCBAD3", "#FFFFD2", "#B5EAD7",
                  "#C7CEEA", "#FFB7B2", "#E2F0CB", "#FFDAC1", "#B5B8FF"]
COULEURS_BLOCS = {
    "Extrême Droite": "#E88D9A", "Droite Traditionnelle": "#7BAFD4",
    "Centre": "#F2CC8F", "Gauche Modérée": "#81B29A", "Gauche Radicale": "#D4A5A5",
}
ORDRE_BLOCS = ["Extrême Droite", "Droite Traditionnelle", "Centre", "Gauche Modérée", "Gauche Radicale"]

plt.rcParams.update({
    'font.family': 'sans-serif', 'font.size': 11, 'font.weight': 'light',
    'axes.titlesize': 13, 'axes.titleweight': 'normal',
    'axes.labelsize': 11, 'axes.labelweight': 'light',
    'axes.spines.top': False, 'axes.spines.right': False,
    'axes.edgecolor': '#CCCCCC', 'axes.grid': True,
    'grid.alpha': 0.3, 'grid.color': '#DDDDDD',
    'legend.frameon': False, 'legend.fontsize': 9,
    'figure.facecolor': 'white', 'axes.facecolor': 'white',
    'xtick.color': '#333333', 'ytick.color': '#333333', 'text.color': '#222222',
})


# Chargement
chemin_propre = "/home/onyxia/work/projet_eco_socio/df_vss_propre.pkl"
if not os.path.exists(chemin_propre):
    chemin_propre = "/home/onyxia/work/projet_eco_socio/dataframes/df_vss_propre.pkl"
df_vss = pd.read_pickle(chemin_propre)
df_vss['date'] = pd.to_datetime(df_vss['date'])
df_blocs = df_vss.dropna(subset=['bloc']).copy()
print(f"{len(df_blocs)} prises de parole chargées ({df_blocs['bloc'].nunique()} blocs).")

7815 prises de parole chargées (5 blocs).


### A.2 Pré-traitement

Le nettoyage est l'étape qui a le plus d'impact sur la qualité des topics. On retire :
- Le **bruit parlementaire** (monsieur, madame, président, amendement, article, commission...)
- Les **mots-clés VSS** eux-memes (sinon ils dominent tous les topics, puisqu'on a filtré le corpus par ces mots)
- On applique le **stemming** (Snowball) et on détecte les **bigrammes** fréquents

In [2]:
stop_words = set(stopwords.words('french'))

# 1. On enrichit considérablement le bruit parlementaire et les tics de langage
bruit = {
    # Bruit parlementaire classique
    "monsieur", "madame", "président", "présidente", "ministre", "mme", "mmes", "m",
    "député", "députée", "députés", "collègue", "collègues",
    "secrétaire", "rapporteur", "rapporteure", "orateur",
    "amendement", "amendements", "article", "articles", "alinéa",
    "projet", "proposition", "loi", "texte", "vote", "voter",
    "adopter", "adopté", "adoptée", "séance", "commission",
    "discussion", "débat", "examen", "lecture", "scrutin",
    "hémicycle", "assemblée", "nationale", "gouvernement",
    "groupe", "banc", "bancs", "applaudissements", "exclamations", "applaudit",
    # Verbes et tics de langage très fréquents
    "demander", "proposer", "souhaiter", "permettre", "falloir",
    "devoir", "pouvoir", "vouloir", "savoir", "croire", "comprendre",
    "penser", "dire", "faire", "être", "avoir", "aller", "peut", "peuvent", "dit", "disent",
    "fait", "faits", "celui", "cela", "ceci", "celle", "celles", "ceux", "cet", "cette",
    "comme", "comment", "autre", "autres", "quelqu", "quelques", "plusieurs", "leur", "leurs",
    "sans", "non", "oui", "très", "bien", "tout", "tous", "toute", "toutes", "trop",
    "aussi", "donc", "encore", "vraiment", "évidemment", "quand",
    "également", "simplement", "effectivement", "certainement", "toujours",
    "aujourd", "hui", "ici", "puis", "enfin", "ainsi", "alors", "car", "parce",
    "premier", "première", "deux", "trois", "million", "milliard", "euro", "euros",
    # Groupes politiques (souvent hurlés dans l'hémicycle)
    "lfi", "nupes", "rn", "lr", "lrem", "em", "modem", "écolo", "ecologistes", "liot", "gdr"
}
stop_words.update([
    'fait', 'peut', 'dire', 'être', 'avoir', 'celui', 'cette', 'tout', 'comme',
    'comment', 'non', 'oui', 'très', 'bien', 'aussi', 'donc', 'encore', 'vraiment',
    'également', 'simplement', 'effectivement', 'certainement', 'toujours',
    'aujourd', 'hui', 'ici', 'puis', 'enfin', 'ainsi', 'alors', 'car', 'parce'
])
bruit.update([
    'rapport', 'commission', 'sénat', 'sénateur', 'député', 'députée',
    'groupe', 'président', 'présidente', 'vice-président', 'secrétaire',
    'questeur', 'bureau', 'conférence', 'ordre du jour', 'scrutin public',
    'vote', 'adopté', 'rejeté', 'amendement', 'sous-amendement',
    'article', 'alinéa', 'projet de loi', 'proposition de loi',
    'lecture', 'nouvelle lecture', 'commission mixte paritaire',
    'cette assemblée', 'cet hémicycle', 'ces bancs'
])
# 2. Les mots de ta requête (VSS) qui masquent les sous-topics
mots_vss = {
    "viol", "viols", "violences", "violence", "sexuel", "sexuelle", "sexuels", "sexuelles",
    "sexiste", "sexistes", "harcèlement", "agression", "agressions", "conjugal", "conjugale",
    "féminicide", "féminicides", "inceste", "outrage", "discrimination", "genre", "sexe",
    "avortement", "ivg", "consentement", "stéréotype", "prostitu", "prostitution",
    "pédocriminalité", "pédophilie", "misogyne", "misogynie", "mutilation", "mutilations",
}

stop_words.update(bruit)
stop_words.update(mots_vss)

def tokeniser(texte):
    # On enlève la ponctuation et on met en minuscules
    texte = re.sub(r'[^a-zàâäéèêëïîôùûüÿçœæ\s]', ' ', str(texte).lower())
    # On filtre les stopwords AVANT le stemming
    return [stemmer.stem(m) for m in texte.split() if m not in stop_words and len(m) > 2]

print("Tokenisation en cours...")
df_blocs['tokens'] = df_blocs['texte'].apply(tokeniser)
df_blocs = df_blocs[df_blocs['tokens'].apply(len) > 3].copy()

# Création des bigrammes
bigram_model = Phrases(df_blocs['tokens'].tolist(), min_count=10, threshold=50)
df_blocs['tokens_bi'] = df_blocs['tokens'].apply(lambda d: bigram_model.freeze()[d])
from gensim.models.phrases import Phraser
bigram_phraser = Phraser(bigram_model)
df_blocs['tokens_bi'] = df_blocs['tokens'].apply(lambda d: bigram_phraser[d])

textes_lda = df_blocs['tokens_bi'].tolist()
dico = corpora.Dictionary(textes_lda)


dico.filter_extremes(no_below=10, no_above=0.1, keep_n=5000)
corpus = [dico.doc2bow(d) for d in textes_lda]

print(f"{len(corpus)} documents, {len(dico)} termes dans le dictionnaire.")

Tokenisation en cours...
7669 documents, 5000 termes dans le dictionnaire.


## Seeded LDA

### Note: Résolution de l'erreur d'installation de GuidedLDA sous Python 3.13

**Le problème d'origine :** La bibliothèque `guidedlda` (dernière mise à jour en 2017) contenait un fichier C pré-compilé (`_guidedlda.c`) qui faisait appel à `longintrepr.h`, un composant interne de Python supprimé depuis Python 3.11. Cela rendait l'installation classique via `pip` impossible sur Python 3.13.

**La solution étape par étape :**
Nous avons forcé la regénération de ce fichier C pour le rendre compatible avec votre système actuel en utilisant Cython.

1. **Clonage du code source :** Récupération des fichiers bruts depuis GitHub au lieu d'utiliser le paquet pré-packagé sur PyPI.
   `git clone https://github.com/vi3k6i5/GuidedLDA`
   `cd GuidedLDA`
2. **Installation de Cython :** L'outil nécessaire pour traduire le code Python/Cython en code C.
   `pip install cython`
3. **Suppression du fichier obsolète :** Retrait de l'ancien fichier C qui bloquait la compilation.
   `rm guidedlda/_guidedlda.c`
4. **Re-génération manuelle du fichier C :** Traduction du fichier source `.pyx` en un nouveau fichier `.c` tout neuf et compatible avec Python 3.13.
   `cython guidedlda/_guidedlda.pyx`
5. **Installation locale :** Installation du paquet en mode "éditable" à partir des sources fraîchement compilées.
   `pip install -e .`

In [8]:
pip install guidedlda

Note: you may need to restart the kernel to use updated packages.


In [9]:
from sklearn.feature_extraction.text import CountVectorizer
import guidedlda
import numpy as np

# Récupérer les textes tokenisés (avant bigrammes) ou après bigrammes
texts_for_lda = df_blocs['tokens_bi'].apply(lambda x: ' '.join(x)).tolist()

# Créer un vectorizer avec le même dictionnaire que gensim (pour garder la cohérence)
vectorizer = CountVectorizer(vocabulary=dico.token2id, lowercase=False)
X = vectorizer.fit_transform(texts_for_lda)   # matrice document-termes (scipy.sparse)

ModuleNotFoundError: No module named 'guidedlda'

In [ ]:
seed_topic_list = [
    ['immigr', 'islam', 'musulman', 'étranger', 'frontière', 
     'clandestin', 'intégration', 'communautarisme', 'voile', 'burqa',
     'racine', 'culture', 'identité', 'nation', 'patrie', 'traditions']
]

In [ ]:
model = guidedlda.GuidedLDA(n_topics=10, n_iter=100, random_state=42, 
                            refresh=20)
model.fit(X, seed_topics=seed_topic_list, seed_confidence=0.2)

In [ ]:
topic_word = model.topic_word_   # shape (n_topics, n_words)
vocab = np.array(vectorizer.get_feature_names_out())
for topic_idx in range(10):
    top_words = vocab[np.argsort(topic_word[topic_idx])[:-11:-1]]
    print(f"Topic {topic_idx}: {', '.join(top_words)}")

In [ ]:
doc_topic = model.transform(X)   # shape (n_docs, n_topics)

In [ ]:

# Ajouter les probabilités au DataFrame
for tid in range(15):
    df_blocs[f'guided_topic_{tid}'] = doc_topic[:, tid]

In [ ]:
from corextopic import corextopic as ct

# Matrice binaire (présence/absence) – CorEx préfère les booléens
X_bin = (X > 0).astype(int)

model = ct.Corex(n_hidden=10, seed=42)
model.fit(X_bin, words=vocab, anchors=seed_topic_list, anchor_strength=3)

# Topics
topics = model.get_topics(n_words=10)
for i, topic in enumerate(topics):
    print(f"Topic {i}: {', '.join(topic)}")

##### A.3 Grid search : nombre de topics et hyperparametres

On teste k de 3 a 25, avec 4 combinaisons alpha x eta, et on évalue chaque modèle
par la cohérence u_mass et c_v. Les résultats sont mis en cache pour ne pas refaire
cette étape couteuse a chaque exécution.

In [ ]:
DOSSIER_DATAFRAMES = "/home/onyxia/work/projet_eco_socio/dataframes"
os.makedirs(DOSSIER_DATAFRAMES, exist_ok=True)
chemin_grid = os.path.join(DOSSIER_DATAFRAMES, "lda_grid_search_results.pkl")

print("La cellule d'exécute effectivement..")
if os.path.exists(chemin_grid):
    with open(chemin_grid, 'rb') as f:
        grid_results = pickle.load(f)
    print(f"Grid search chargée depuis le cache ({len(grid_results)} configs testées).")
else:
    print("Grid search en cours (10-30 min)...\n")
    topics_range = [7, 9, 12, 13, 15, 17, 20, 25]
    grid_results = []
    total = len(topics_range) * 4
    idx = 0
    for k in topics_range:
        for alpha in ['symmetric', 'asymmetric']:
            for eta in ['symmetric', 'auto']:
                idx += 1
                print(f"  [{idx}/{total}] k={k}, alpha={alpha}, eta={eta}...", end=" ")
                try:
                    m = LdaMulticore(corpus=corpus, id2word=dico, num_topics=k,
                                     alpha=alpha, eta=eta, passes=15, iterations=200,
                                     random_state=42, chunksize=500, per_word_topics=True)
                    s_u = CoherenceModel(model=m, corpus=corpus, dictionary=dico, coherence='u_mass').get_coherence()
                    s_c = CoherenceModel(model=m, texts=textes_lda, dictionary=dico, coherence='c_v').get_coherence()
                    p = m.log_perplexity(corpus)
                    grid_results.append({'k':k,'alpha':alpha,'eta':eta,'u_mass':s_u,'c_v':s_c,'perplexity':p})
                    print(f"u_mass={s_u:.4f}, c_v={s_c:.4f}")
                except Exception as e:
                    print(f"ERREUR: {e}")
    with open(chemin_grid, 'wb') as f:
        pickle.dump(grid_results, f)
    print(f"\nSauvegardé dans {chemin_grid}")

### A.4 Visualisation des scores de cohérence

In [ ]:
df_grid = pd.DataFrame(grid_results)

# Compatibilite avec l'ancien cache (noms de colonnes differents)
if 'coherence_umass' in df_grid.columns and 'u_mass' not in df_grid.columns:
    df_grid = df_grid.rename(columns={'coherence_umass': 'u_mass', 'coherence_cv': 'c_v'})
    print("Colonnes renommees depuis l'ancien format du cache.")

fig, axes = plt.subplots(1, 2, figsize=(15, 5))
for (alpha, eta), grp in df_grid.groupby(['alpha', 'eta']):
    g = grp.sort_values('k')
    axes[0].plot(g['k'], g['u_mass'], marker='o', label=f"a={alpha}, e={eta}", lw=1.5)
    axes[1].plot(g['k'], g['c_v'], marker='s', label=f"a={alpha}, e={eta}", lw=1.5)
axes[0].set_xlabel("Nombre de topics (k)")
axes[0].set_ylabel("Coherence u_mass (plus c'est haut, mieux c'est)")
axes[0].set_title("Score u_mass")
axes[0].legend(fontsize=8)
axes[1].set_xlabel("Nombre de topics (k)")
axes[1].set_ylabel("Coherence c_v")
axes[1].set_title("Score c_v")
axes[1].legend(fontsize=8)
plt.suptitle("Selection du nombre optimal de topics", fontweight='bold')
plt.tight_layout()
plt.show()

print("\nTop 5 par u_mass :")
print(df_grid.sort_values('u_mass', ascending=False).head(5).to_string(index=False))

### A.5 Sélection du modèle optimal

On ne retient pas le k=3 meme s'il a le meilleur score de cohérence. Avec seulement
3 topics, on ne peut pas distinguer un signal identitaire fin dans un corpus thématiquement
riche (VSS couvre la PMA, le harcèlement au travail, les violences conjugales, l'IVG, etc.).

On impose un **plancher a 12 topics** et on sélectionne la meilleure configuration au-dessus.
Ce choix est justifié par le fait que notre corpus mélange au moins 5-6 sous-thématiques
très distinctes, et qu'on cherche un signal spécifique (immigration/identité) qui ne peut
émerger que dans un modèle suffisamment granulaire.

In [ ]:
# Compatibilite noms de colonnes
if 'coherence_umass' in df_grid.columns and 'u_mass' not in df_grid.columns:
    df_grid = df_grid.rename(columns={'coherence_umass': 'u_mass', 'coherence_cv': 'c_v'})

# Filtrer : on garde seulement k >= 12
df_filtered = df_grid[df_grid['k'] >= 12].copy()
best = df_filtered.sort_values('u_mass', ascending=False).iloc[0]
best_k = int(best['k'])

print(f"Meilleure config avec k >= 12 :")
print(f"  k={best_k}, alpha={best['alpha']}, eta={best['eta']}")
print(f"  u_mass={best['u_mass']:.4f}, c_v={best['c_v']:.4f}")

best_global = df_grid.sort_values('u_mass', ascending=False).iloc[0]
print(f"\n(Pour info, le meilleur global est k={int(best_global['k'])} avec u_mass={best_global['u_mass']:.4f})")
if int(best_global['k']) < 12:
    print(f"On ne le retient pas car {int(best_global['k'])} topics est trop grossier pour notre analyse.)")

chemin_lda = os.path.join(DOSSIER_DATAFRAMES, "lda_best_model.pkl")
chemin_dict = os.path.join(DOSSIER_DATAFRAMES, "lda_best_dict.pkl")

# Verification : si le modele en cache a un k different de best_k, on le supprime
if os.path.exists(chemin_lda):
    try:
        from gensim.models import LdaModel as _LDA
        _cached = _LDA.load(chemin_lda)
        if _cached.num_topics != best_k:
            print(f"\nLe modele en cache a {_cached.num_topics} topics, mais on veut {best_k}.")
            print("Suppression du cache pour reentrainer...")
            os.remove(chemin_lda)
            if os.path.exists(chemin_dict):
                os.remove(chemin_dict)
            # Supprimer aussi les fichiers annexes de gensim
            for ext in ['.state', '.id2word', '.expElogbeta.npy']:
                p = chemin_lda + ext
                if os.path.exists(p): os.remove(p)
        del _cached
    except Exception:
        pass

if os.path.exists(chemin_lda):
    lda_best = LdaModel.load(chemin_lda)
    dico = corpora.Dictionary.load(chemin_dict)
    corpus = [dico.doc2bow(d) for d in textes_lda]
    print(f"Modele LDA charge depuis le cache (k={lda_best.num_topics}).")
else:
    print(f"\nEntrainement du modele final (k={best_k}, passes=30, iterations=400)...")
    lda_best = LdaMulticore(corpus=corpus, id2word=dico, num_topics=best_k,
                             alpha=best['alpha'], eta=best['eta'],
                             passes=30, iterations=400, random_state=42,
                             chunksize=500, per_word_topics=True)
    lda_best.save(chemin_lda)
    dico.save(chemin_dict)
    print(f"Sauvegarde (k={best_k}).")

### A.6 Inspection des topics

In [ ]:
print("=" * 80)
print(f"TOPICS (k={lda_best.num_topics})")
print("=" * 80)

tc = CoherenceModel(model=lda_best, texts=textes_lda, dictionary=dico,
                    coherence='u_mass', topn=12).get_coherence_per_topic()

for tid in range(lda_best.num_topics):
    mots = lda_best.show_topic(tid, topn=12)
    mots_str = ", ".join([f"{m} ({p:.3f})" for m, p in mots])
    c = tc[tid] if tid < len(tc) else float('nan')
    print(f"\n  Topic #{tid:2d} [coherence={c:+.4f}]")
    print(f"    {mots_str}")

### A.7 Similarité inter-topics et distribution par bloc

In [ ]:
from sklearn.metrics.pairwise import cosine_similarity as cos_sim

tw = lda_best.get_topics()
sim = cos_sim(tw)

fig, ax = plt.subplots(figsize=(8, 7))
mask = np.eye(sim.shape[0], dtype=bool)
sns.heatmap(sim, annot=True, fmt=".2f", cmap="RdPu", mask=mask, vmin=0, vmax=0.5,
            linewidths=0.3, linecolor='white',
            xticklabels=[f"T{i}" for i in range(sim.shape[0])],
            yticklabels=[f"T{i}" for i in range(sim.shape[0])], ax=ax)
ax.set_title("Similarité cosinus inter-topics")
plt.tight_layout()
plt.show()

In [ ]:
# Distribution par bloc
dists = []
for bow in tqdm(corpus, desc="Inference"):
    dt = lda_best.get_document_topics(bow, minimum_probability=0.0)
    probs = [0.0] * lda_best.num_topics
    for tid, p in dt: probs[tid] = p
    dists.append(probs)

df_topics = pd.DataFrame(dists, columns=[f"topic_{i}" for i in range(lda_best.num_topics)])
df_topics['bloc'] = df_blocs['bloc'].values[:len(df_topics)]
df_topics['date'] = df_blocs['date'].values[:len(df_topics)]

tc = [f"topic_{i}" for i in range(lda_best.num_topics)]
moy = df_topics.groupby('bloc')[tc].mean()
moy = moy.reindex([b for b in ORDRE_BLOCS if b in moy.index])

fig, ax = plt.subplots(figsize=(13, 6))
sns.heatmap(moy, annot=True, fmt=".3f", cmap="BuPu", linewidths=0.3, linecolor='white',
            cbar_kws={'label': 'Probabilité moyenne', 'shrink': 0.8}, ax=ax)
ax.set_title("Distribution moyenne des topics par bloc")
ax.set_xlabel("Topic")
ax.set_ylabel("")
plt.tight_layout()
plt.show()

### A.8 Evolution temporelle d'un topic

Après avoir inspecté les topics, identifiez celui qui contient des termes liés
a l'immigration/identité et mettez son numéro dans `TOPIC_A_SUIVRE`.
Si aucun topic ne correspond, c'est un résultat en soi : la LDA ne capte pas ce signal.

In [ ]:
TOPIC_A_SUIVRE = 0  # a ajuster apres inspection
NOM_TOPIC = "Topic a identifier"

# Conversion en annee (int, pas string, pour eviter les problemes de tri)
df_topics['annee'] = df_topics['date'].dt.year
col = f"topic_{TOPIC_A_SUIVRE}"
evol = df_topics.groupby(['annee', 'bloc'])[col].mean().reset_index()

fig, ax = plt.subplots(figsize=(11, 5))
for bloc in ORDRE_BLOCS:
    s = evol[evol['bloc']==bloc].sort_values('annee')
    if s.empty: continue
    ax.plot(s['annee'], s[col], marker='o', label=bloc, color=COULEURS_BLOCS[bloc], lw=2)
ax.set_title(f"Evolution du {NOM_TOPIC} (topic #{TOPIC_A_SUIVRE}) par bloc")
ax.set_ylabel("Probabilite moyenne")
ax.set_xlabel("Annee")
ax.legend(bbox_to_anchor=(1.01, 1), loc='upper left')
plt.tight_layout()
plt.show()

## Partie B . Embeddings Sentence-CamemBERT et similarité cosinus

### Pourquoi Sentence-CamemBERT et pas CamemBERT brut ?

Le problème de la version précédente venait du fait qu'on utilisait **CamemBERT-base** avec
le **vecteur CLS** pour représenter des documents entiers. Comme l'ont montré Reimers et
Gurevych (2019), les modèles BERT/CamemBERT non fine-tunés produisent des représentations
CLS qui ne sont **pas adaptées a la similarité cosinus**. Tous les documents se retrouvent
dans une region très concentrée de l'espace, d'ou les scores de ~0.99 observés.

**Sentence-CamemBERT-Large** (La Javaness, 2022) résout ce problème. C'est un CamemBERT
fine-tuné avec un réseau siamois (architecture Sentence-BERT) sur le dataset STSb francais.
Il utilise le **mean pooling** (moyenne de tous les tokens) au lieu du CLS token,
et l'espace d'embeddings résultant est spécifiquement optimisé pour que la similarité cosinus
reflète la proximité sémantique réelle.

Ce modèle est classé parmi les meilleurs pour le francais sur le benchmark MTEB-French
(Ciancone et al., 2024).

### B.1 Installation et chargement du modèle

In [ ]:
import subprocess, sys

try:
    from sentence_transformers import SentenceTransformer
except ImportError:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "sentence-transformers"])
    from sentence_transformers import SentenceTransformer

try:
    import torch
    print(f"GPU disponible : {torch.cuda.is_available()}")
except:
    pass

# On utilise le modèle Sentence-CamemBERT-Large (La Javaness)
# Fine-tuné sur STSb pour la similarité cosinus
MODEL_NAME = "dangvantuan/sentence-camembert-large"
print(f"Chargement de {MODEL_NAME}...")
st_model = SentenceTransformer(MODEL_NAME)
print(f"Modèle chargé. Dimension des embeddings : {st_model.get_sentence_embedding_dimension()}")

### B.2 Encodage des prises de parole

Le modèle Sentence-CamemBERT encode directement une phrase ou un paragraphe en un vecteur
de 1024 dimensions, optimisé pour la similarité cosinus. On encode toutes les prises de parole
du corpus et on sauvegarde le résultat.

In [ ]:
chemin_emb = "/home/onyxia/work/projet_eco_socio/df_vss_embeddings_sentence.pkl"

if os.path.exists(chemin_emb):
    df_blocs = pd.read_pickle(chemin_emb)
    print(f"Embeddings chargés depuis le cache : {len(df_blocs)} prises de parole.")
else:
    print(f"Encodage de {len(df_blocs)} prises de parole...")
    print("(Sentence-CamemBERT-Large\n")

    # On encode par batches pour la barre de progression
    textes = df_blocs['texte'].tolist()
    # Troncature des textes trop longs (le modele accepte max 512 tokens)
    # Certains discours parlementaires depassent largement cette limite.
    # On tronque a ~1500 caracteres (~400 tokens) pour rester dans les limites.
    textes_tronques = [t[:1500] if len(t) > 1500 else t for t in textes]
    print(f"   {sum(1 for t in textes if len(t) > 1500)} textes tronques (> 1500 caracteres)")

    embeddings = st_model.encode(textes_tronques, batch_size=32, show_progress_bar=True,
                                  normalize_embeddings=True)
    df_blocs['vecteur'] = list(embeddings)

    df_blocs.to_pickle(chemin_emb)
    print(f"\nSauvegardé dans {chemin_emb}")

### B.3 Similarité cosinus entre blocs et extrême droite

Pour chaque année, on calcule le **centroide** de chaque bloc (moyenne de tous les vecteurs
du bloc pour cette année), puis la similarité cosinus entre chaque centroide et celui
de l'extreme droite.

Comme les embeddings sont normalisés en L2 (norme = 1), la similarité cosinus revient
a un simple produit scalaire entre centroïdes.

In [ ]:
from sklearn.metrics.pairwise import cosine_similarity

df_blocs['annee'] = df_blocs['date'].dt.year

resultats = []
for annee in sorted(df_blocs['annee'].unique()):
    df_an = df_blocs[df_blocs['annee'] == annee]
    if "Extrême Droite" not in df_an['bloc'].values:
        continue
    vecs_ed = np.vstack(df_an[df_an['bloc']=="Extrême Droite"]['vecteur'].values)
    cent_ed = np.mean(vecs_ed, axis=0)
    for bloc in df_an['bloc'].unique():
        if bloc == "Extrême Droite": continue
        vecs = np.vstack(df_an[df_an['bloc']==bloc]['vecteur'].values)
        cent = np.mean(vecs, axis=0)
        sim = cosine_similarity([cent_ed], [cent])[0][0]
        resultats.append({'annee': annee, 'bloc': bloc, 'similarite': sim})

df_sim = pd.DataFrame(resultats)

pal = {b: COULEURS_BLOCS[b] for b in COULEURS_BLOCS if b != "Extrême Droite"}
fig, ax = plt.subplots(figsize=(11, 6))
sns.lineplot(data=df_sim, x='annee', y='similarite', hue='bloc',
             palette=pal, marker='o', lw=2, ax=ax)
ax.set_title("Proximité sémantique des blocs avec l'extreme droite\n(Sentence-CamemBERT-Large, centroïdes annuels)")
ax.set_ylabel("Similarité cosinus")
ax.set_xlabel("Année")
ax.legend(title="Bloc", bbox_to_anchor=(1.01, 1), loc='upper left')
ylim_min = max(0, df_sim['similarite'].min() - 0.05)
ax.set_ylim(ylim_min, min(1.0, df_sim['similarite'].max() + 0.02))
plt.tight_layout()
plt.show()

print("\nSimilarités moyennes sur la période :")
print(df_sim.groupby('bloc')['similarite'].mean().sort_values(ascending=False).to_string())

### B.4 Convergence vers le concept "immigration/identité"

On encode une phrase-concept avec le meme modèle, puis on mesure comment chaque bloc
s'en rapproche au fil du temps.

In [ ]:
phrase_concept = (
    "immigration clandestin frontière étranger identité nationale "
    "culture expulsion OQTF territoire islam intégration"
)
vec_concept = st_model.encode([phrase_concept], normalize_embeddings=True)[0]

df_blocs['prox_concept'] = df_blocs['vecteur'].apply(
    lambda v: cosine_similarity([v], [vec_concept])[0][0]
)

df_derive = df_blocs.groupby(['annee', 'bloc'])['prox_concept'].mean().reset_index()

fig, ax = plt.subplots(figsize=(11, 6))
for bloc in ORDRE_BLOCS:
    s = df_derive[df_derive['bloc']==bloc].sort_values('annee')
    if s.empty: continue
    ax.plot(s['annee'], s['prox_concept'], marker='o', label=bloc,
            color=COULEURS_BLOCS[bloc], lw=2.5, markersize=6)
ax.set_title("Convergence vers le cadrage 'immigration/identité'\n(similarité cosinus avec le concept de référence)")
ax.set_ylabel("Similarité avec le concept identitaire")
ax.set_xlabel("Année")
ax.legend(title="Bloc", bbox_to_anchor=(1.01, 1), loc='upper left')
plt.tight_layout()
plt.show()

print("\nProximité moyenne au concept identitaire :")
print(df_blocs.groupby('bloc')['prox_concept'].mean().sort_values(ascending=False).to_string())

### B.5 Distribution des similarités par bloc (boxplots)

Pour aller au-dela des moyennes, on regarde la **distribution** des similarités
au concept identitaire pour chaque bloc. Si l'extreme droite a non seulement
une moyenne plus élevée mais aussi une queue de distribution vers le haut,
c'est un signal plus robuste.

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
data_plot = df_blocs[df_blocs['bloc'].isin(ORDRE_BLOCS)].copy()
data_plot['bloc'] = pd.Categorical(data_plot['bloc'], categories=ORDRE_BLOCS, ordered=True)

sns.boxplot(data=data_plot, x='bloc', y='prox_concept',
            palette=COULEURS_BLOCS, ax=ax, fliersize=2, linewidth=0.8)
ax.set_title("Distribution de la proximité au concept identitaire par bloc")
ax.set_ylabel("Similarité cosinus avec le concept immigration/identité")
ax.set_xlabel("")
plt.xticks(rotation=15, ha='right')
plt.tight_layout()
plt.show()

## Références

- Blei, D. M., Ng, A. Y., Jordan, M. I. (2003). Latent Dirichlet Allocation. JMLR.
- Mimno, D. et al. (2011). Optimizing Semantic Coherence in Topic Models. EMNLP.
- Röder, M. et al. (2015). Exploring the Space of Topic Coherence Measures. WSDM.
- Reimers, N., Gurevych, I. (2019). Sentence-BERT: Sentence Embeddings using Siamese BERT-Networks. EMNLP.
- Martin, L. et al. (2020). CamemBERT: a Tasty French Language Model. ACL.
- La Javaness (2022). Sentence Embedding Fine-tuning for the French Language.
- Ciancone, A. et al. (2024). MTEB-French: Resources for French Sentence Embedding Evaluation.